# 🍷 Wine Quality Prediction

**Oasis Infobyte Data Analytics — Task 2**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
df = pd.read_csv("data/WineQT.csv")
print("Shape:", df.shape)
display(df.head())
df.info()


In [ ]:
display(df["quality"].value_counts().sort_index())
sns.countplot(data=df, x="quality")
plt.title("Quality Score Distribution")
plt.show()

print("Class imbalance: quality 5 and 6 dominate, while 3, 4 and 8 are rare.")


In [ ]:
df.drop(columns=["Id", "quality"]).hist(figsize=(14,10), bins=20)
plt.suptitle("Physicochemical Feature Distributions")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df.drop(columns=["Id"]).corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
# Binary feature engineering:
# quality >= 7 -> Good Wine
# quality < 7  -> Lower Quality
df["good_wine"] = (df["quality"] >= 7).astype(int)
display(df["good_wine"].value_counts())

print("Binary grouping is used because the original quality classes are imbalanced.")


In [ ]:
X = df.drop(columns=["quality", "good_wine", "Id"])
y = df["good_wine"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)


In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced"
    ),
    "SGD": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SGDClassifier(
            loss="log_loss", max_iter=2000, random_state=42, class_weight="balanced"
        ))
    ]),
    "SVC": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", class_weight="balanced"))
    ])
}


In [ ]:
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = accuracy_score(y_test, pred)

    print("\n" + name)
    print("Accuracy:", f"{results[name]:.4f}")
    print(classification_report(y_test, pred, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, pred))


In [ ]:
comparison = pd.DataFrame({"Accuracy": results}).sort_values(
    "Accuracy", ascending=False
)
display(comparison)

importance = pd.Series(
    models["Random Forest"].feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importance.head(10).sort_values().plot(kind="barh", figsize=(10,6))
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.show()


##  Conclusion

Three classification models were trained and compared: Random Forest, SGD, and Support Vector Classifier (SVC).

The model performance was:

- Random Forest: 91.27% accuracy
- SVC: 81.66% accuracy
- SGD: 71.62% accuracy

Among the three models, **Random Forest is the most suitable for deployment** because it achieved the highest accuracy of **91.27%**. It also provides feature importance values, which makes the model easier to interpret and understand.

Therefore, Random Forest is selected as the final model for the Wine Quality Prediction application.
